# Portfolio Project 2, Part 1: An Intent Classifier for My Hotel Booking Chatbot

**Course:** AITCD001 Chatbots Development  
**Dataset:** Assignment 8 Hotel Booking Chatbot prototype  
**Status:** Provisional experiment using the synthetic Assignment 8 dataset. The original Portfolio Project 1 chatbot rules have not been verified.

This notebook follows the lecturer's Part 1 sequence: load the labelled dataset, score the rules on the same test set, compare word and character features, evaluate every intent, inspect the confusion matrix, select a confidence threshold, save the model, and try new messages.

**Important limitation:** the current Assignment 8 CSV marks all 180 messages as `synthetic`. This notebook's rule baseline is copied from the provisional keyword rules in the Assignment 8 prototype, not verified rules from the original Portfolio Project 1 source code. The results are therefore provisional and should not be represented as verified original-chatbot performance.

---
## Setup

The provided classifier template uses TF-IDF features and Logistic Regression. The character model uses character-boundary n-grams (2–4).

In [1]:
import re
import joblib
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

print("Libraries ready")

Libraries ready


---
## Step 1: My dataset

I use the `intents.csv` produced from the Assignment 8 prototype. It contains a stratified 80/20 split: 144 training messages and 36 test messages, with six messages per intent in the test set. The split is stored in the CSV so that the rules and classifier use exactly the same test messages.

In [2]:
BOT_NAME = "Provisional Hotel Booking Chatbot"
FILE = "intents.csv"
df = pd.read_csv(FILE)
train = df[df["split"] == "train"].copy()
test = df[df["split"] == "test"].copy()
train_texts = train["text"].astype(str).tolist()
train_labels = train["intent"].tolist()
test_texts = test["text"].astype(str).tolist()
test_labels = test["intent"].tolist()
labels = sorted(df["intent"].unique())

print(BOT_NAME)
print("Rows:", len(df), "| training:", len(train), "| test:", len(test))
print("Labels:", labels)
print("Source counts:")
print(df["source"].value_counts().to_string())
print("Test messages per intent:")
print(test["intent"].value_counts().sort_index().to_string())

Provisional Hotel Booking Chatbot
Rows: 180 | training: 144 | test: 36
Labels: ['booking_cancellation', 'booking_modification', 'hotel_amenities', 'out_of_scope', 'room_availability', 'room_booking']
Source counts:
source
synthetic    180
Test messages per intent:
intent
booking_cancellation    6
booking_modification    6
hotel_amenities         6
out_of_scope            6
room_availability       6
room_booking            6


---
## Step 2: The rules, as the provisional baseline

The original Portfolio Project 1 source code and rules were not present in the repository when this notebook was prepared. I therefore use the provisional rules from the Assignment 8 prototype only. The first matching rule wins; if none matches, the message is labelled `out_of_scope`. This is **not** a verified baseline for the original chatbot.

In [3]:
RULES = [
    ("booking_cancellation", r"\bcancel|cancell|hagarika|guhagarika|gusiba|isibwa|nyihagarike|withdraw|terminate\b"),
    ("booking_modification", r"\bchange|modify|update|amend|move my|extend|shift|guhindura|kongera|edit|add one more\b"),
    ("room_availability", r"\bavailable|availability|vacan|free|open|iraboneka|zihari|ihari|vacancy|fully booked|check avail\b"),
    ("hotel_amenities", r"\bwi-?fi|wifi|internet|pool|piscine|breakfast|parking|restaurant|gym|facilit|laundry|shuttle|air conditioning|service|towels|elevator|lift|hot water|salle de sport|generator|tv\b"),
    ("room_booking", r"\bbook|booking|reserve|reservation|fata|kubika|kubookinga|bika|room for|accommodation\b"),
]

def rule_label(message):
    text = str(message).lower()
    for label, pattern in RULES:
        if re.search(pattern, text):
            return label
    return "out_of_scope"

def outcomes(guesses, actual):
    right = missed = wrong = 0
    for guess, label in zip(guesses, actual):
        if guess == label:
            right += 1
        elif guess == "out_of_scope":
            missed += 1
        else:
            wrong += 1
    return right, missed, wrong

rule_predictions = [rule_label(message) for message in test_texts]
rules_result = outcomes(rule_predictions, test_labels)
print("Provisional rules:", len(RULES))
print("Rules on the shared test set — right, missed, wrong:", rules_result)
print(f"Rules accuracy: {accuracy_score(test_labels, rule_predictions):.1%}")

Provisional rules: 5
Rules on the shared test set — right, missed, wrong: (32, 1, 3)
Rules accuracy: 88.9%


---
## Step 3: Words and characters

Both models are trained on the training set only. The model with the higher test accuracy is kept. These results use a small synthetic dataset, so they should be treated as an initial experiment rather than evidence of real-world performance.

In [4]:
CANDIDATES = {
    "words": TfidfVectorizer(),
    "characters": TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 4)),
}
accuracy = {}
models = {}
for name, vectorizer in CANDIDATES.items():
    model = make_pipeline(vectorizer, LogisticRegression(max_iter=1000, random_state=42))
    model.fit(train_texts, train_labels)
    models[name] = model
    accuracy[name] = accuracy_score(test_labels, model.predict(test_texts))
    print(f"{name:11} accuracy {accuracy[name]:.1%}")

best = max(accuracy, key=accuracy.get)
classifier = models[best]
predicted = classifier.predict(test_texts)
print("Kept:", best)

words       accuracy 80.6%


characters  accuracy 86.1%
Kept: characters


---
## Step 4: Every intent, and the confusion matrix

Precision measures how often a predicted intent is correct. Recall measures how many messages of an intent are found. F1 balances precision and recall. The confusion matrix shows the actual labels against the predicted labels.

In [5]:
print(classification_report(test_labels, predicted, labels=labels, zero_division=0, digits=3))
print("Confusion matrix (rows = actual, columns = predicted):")
print(pd.crosstab(pd.Series(test_labels, name="actual"),
                  pd.Series(predicted, name="predicted"))
        .reindex(index=labels, columns=labels, fill_value=0).to_string())

                      precision    recall  f1-score   support

booking_cancellation      1.000     0.833     0.909         6
booking_modification      0.833     0.833     0.833         6
     hotel_amenities      0.857     1.000     0.923         6
        out_of_scope      0.800     0.667     0.727         6
   room_availability      1.000     1.000     1.000         6
        room_booking      0.714     0.833     0.769         6

            accuracy                          0.861        36
           macro avg      0.867     0.861     0.860        36
        weighted avg      0.867     0.861     0.860        36

Confusion matrix (rows = actual, columns = predicted):
predicted             booking_cancellation  booking_modification  hotel_amenities  out_of_scope  room_availability  room_booking
actual                                                                                                                          
booking_cancellation                     5                     1

---
## Step 5: The threshold table

Below the threshold, the classifier falls back to `out_of_scope`. Each row reports correct answers, total fallbacks, real requests missed by the fallback, and confident wrong predictions.

In [6]:
scores = classifier.predict_proba(test_texts)
classes = classifier.classes_
print("threshold  right  fallbacks  missed  wrong")
for threshold in [0.0, 0.1, 0.15, 0.2, 0.25, 0.3, 0.4, 0.5]:
    final = []
    for row in scores:
        best_i = row.argmax()
        intent = classes[best_i]
        if row[best_i] < threshold:
            intent = "out_of_scope"
        final.append(intent)
    right, missed, wrong = outcomes(final, test_labels)
    fallbacks = final.count("out_of_scope")
    print(f"{threshold:9.2f} {right:6} {fallbacks:10} {missed:7} {wrong:6}")

threshold  right  fallbacks  missed  wrong
     0.00     31          5       1      4
     0.10     31          5       1      4
     0.15     31          5       1      4
     0.20     31          5       1      4
     0.25     27          9       5      4
     0.30     20         22      16      0
     0.40     10         32      26      0
     0.50      6         36      30      0


---
## Step 6: Choose the threshold, and say why

For a hotel booking chatbot, a confident but incorrect booking-related answer could cause a guest to arrive on the wrong date or with the wrong room details. A fallback asks the guest to clarify or contact the hotel, which is inconvenient but safer. I choose **0.30** for this provisional experiment because the threshold table shows zero confident wrong predictions on this test set. The trade-off is substantial: 22 of 36 test messages fall back (61.1%); 16 are in-scope requests missed by the fallback. This threshold does not make the classifier better than the provisional rules; it prioritises avoiding wrong answers over answering more messages.

In [7]:
THRESHOLD = 0.30
WHY = ("I chose 0.30 because an incorrect hotel booking answer can cause a guest to "
       "receive the wrong reservation details. A fallback is inconvenient, but safer. "
       "On this small test set it gives zero confident wrong predictions, while 16 of 36 "
       "messages fall back. This is a conservative trade-off, not evidence that the model "
       "outperforms the provisional rules.")
final_predictions = []
for row in classifier.predict_proba(test_texts):
    best_i = row.argmax()
    intent = classifier.classes_[best_i]
    if row[best_i] < THRESHOLD:
        intent = "out_of_scope"
    final_predictions.append(intent)
model_result = outcomes(final_predictions, test_labels)
fallback_rate = final_predictions.count("out_of_scope") / len(final_predictions)
print("Threshold:", THRESHOLD)
print(WHY)
print("                  right  missed  wrong")
print(f"Provisional rules {rules_result[0]:6} {rules_result[1]:7} {rules_result[2]:6}")
print(f"Classifier        {model_result[0]:6} {model_result[1]:7} {model_result[2]:6}")
print(f"Classifier fallback rate: {fallback_rate:.1%}")

Threshold: 0.3
I chose 0.30 because an incorrect hotel booking answer can cause a guest to receive the wrong reservation details. A fallback is inconvenient, but safer. On this small test set it gives zero confident wrong predictions, while 16 of 36 messages fall back. This is a conservative trade-off, not evidence that the model outperforms the provisional rules.
                  right  missed  wrong
Provisional rules     32       1      3
Classifier            20      16      0
Classifier fallback rate: 61.1%


---
## Step 7: Save the classifier and try new messages

The saved pipeline includes both the TF-IDF vectorizer and Logistic Regression model. The four test messages below are illustrative examples, not a substitute for testing with real hotel guests.

In [8]:
# float32 coefficients reduce the saved file size; the measured predictions are unchanged to rounding precision.
classifier.steps[-1][1].coef_ = classifier.steps[-1][1].coef_.astype("float32")
classifier.steps[-1][1].intercept_ = classifier.steps[-1][1].intercept_.astype("float32")
joblib.dump(classifier, "intent_classifier.joblib", compress=9)
loaded = joblib.load("intent_classifier.joblib")
TRY = [
    "Do you have a family room free tmrw?",
    "pls cancel my booking for friday",
    "Can I get breakfast and free wifi?",
    "what is the weather in Kigali today?",
]
for message in TRY:
    row = loaded.predict_proba([message])[0]
    best_i = row.argmax()
    intent = loaded.classes_[best_i]
    confidence = row[best_i]
    verdict = "answer" if confidence >= THRESHOLD and intent != "out_of_scope" else "fallback"
    print(f"{message:45} {intent:23} {confidence:.2f} {verdict}")
print("Saved intent_classifier.joblib")

Do you have a family room free tmrw?          room_availability       0.37 answer
pls cancel my booking for friday              booking_cancellation    0.52 answer
Can I get breakfast and free wifi?            hotel_amenities         0.43 answer
what is the weather in Kigali today?          out_of_scope            0.44 fallback
Saved intent_classifier.joblib


---
## Step 8: README and publication

The matching `README.md` records the actual model accuracies, per-intent metrics, threshold decision, and same-test-set comparison. The notebook, split CSV, README, and saved model belong together in the `dataset/` folder of the Portfolio Project 1 repository.

**Limitations to retain in the submitted README:** all dataset messages are synthetic; the original Portfolio Project 1 rules have not been verified; and the provisional classifier does not outperform the provisional rules at the selected threshold. The next improvement is to use the lecturer's Assignment 8 feedback and genuine, anonymised guest messages, then repeat the evaluation with the actual chatbot rules.